In [1]:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft triton -q
!pip install "datasets>=3.4.1,<4.0.0" -q
!pip install transformers==4.55.4 trl==0.22.2 unsloth unsloth_zoo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_sys

In [2]:
import unsloth
import os, torch, sys, json, re, random
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

SYSTEM_PROMPT = (
    "You are a math reasoning tutor. "
    "Solve the problem and give the final numeric answer after ####."
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-14 18:16:16.159878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765736176.405156      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765736176.470049      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

🦥 Unsloth Zoo will now patch everything to make training faster!


# Load Model

In [3]:
model_name = "unsloth/mistral-7b-instruct-v0.3"
max_seq_length = 2048

print("Loading Mistral 7B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=False,
    device_map="cuda:0",
    trust_remote_code=True,
)
print("Loaded successfully.")

Loading Mistral 7B...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.5: Fast Mistral patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded successfully.


# Load Dataset

In [4]:
print("Loading GSM8K train...")
gsm = load_dataset("openai/gsm8k", "main")
train_raw = gsm["train"].shuffle(seed=SEED)

EVAL_SIZE = 500
SFT_SIZE  = 4000
RL_SIZE   = 2900

eval_raw = train_raw.select(range(EVAL_SIZE))

remaining = train_raw.select(range(EVAL_SIZE, EVAL_SIZE + SFT_SIZE + RL_SIZE))
sft_raw = remaining.select(range(SFT_SIZE))
rl_raw  = remaining.select(range(SFT_SIZE, SFT_SIZE + RL_SIZE))

print("Eval =", len(eval_raw))
print("SFT  =", len(sft_raw))
print("RL   =", len(rl_raw))

Loading GSM8K train...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Eval = 500
SFT  = 4000
RL   = 2900


In [5]:
def extract_gold(a):
    return a.split("####")[-1].strip() if "####" in a else a.strip()

def to_df(split):
    rows = []
    for ex in split:
        rows.append({
            "question": ex["question"],
            "full_answer": ex["answer"],
            "gold_answer": extract_gold(ex["answer"]),
        })
    return pd.DataFrame(rows)

eval_df = to_df(eval_raw)
sft_df  = to_df(sft_raw)
rl_df   = to_df(rl_raw)

eval_df.to_csv("gsm8k_eval_500.csv", index=False)
sft_df.to_csv("gsm8k_sft_4000.csv", index=False)
rl_df.to_csv("gsm8k_rl_2900.csv", index=False)

print("Saved all CSV splits.")

Saved all CSV splits.


# Build CoT Dataset

In [6]:
EOS_TOKEN = tokenizer.eos_token

def build_chat_cot_with_eos(df):
    """Build chat format with EOS token appended"""
    msgs_list = []
    for _, row in df.iterrows():
        msgs_list.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": row["question"]},
            {"role": "assistant", "content": row["full_answer"] + EOS_TOKEN},
        ])
    return Dataset.from_dict({"messages": msgs_list})

def format_to_text(batch):
    """Format messages to text using chat template"""
    texts = []
    for msgs in batch["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, 
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

def filter_by_length(ds, tokenizer, max_len=1536):
    """Filter dataset by token length"""
    def compute_length(batch):
        lens = tokenizer(
            batch["text"], 
            return_length=True, 
            add_special_tokens=False
        )["length"]
        return {"length": lens}
    
    ds = ds.map(compute_length, batched=True, batch_size=32)
    initial_size = len(ds)
    ds = ds.filter(lambda x: x["length"] <= max_len)
    filtered_size = len(ds)
    print(f"Filtered: {initial_size} -> {filtered_size} samples (removed {initial_size - filtered_size})")
    return ds

print("Building SFT dataset with EOS token...")
sft_chat = build_chat_cot_with_eos(sft_df)
sft_chat = sft_chat.map(format_to_text, batched=True, batch_size=32)
sft_chat = filter_by_length(sft_chat, tokenizer, max_len=1536)

sft_text = sft_chat.map(
    lambda x: {"text": x["text"]},
    remove_columns=sft_chat.column_names,
)

print("Final SFT samples:", len(sft_text))
print("\nExample formatted text:")
print(sft_text[0]["text"][:500] + "...")

Building SFT dataset with EOS token...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4000 [00:00<?, ? examples/s]

Filtered: 4000 -> 4000 samples (removed 0)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Final SFT samples: 4000

Example formatted text:
<s>[INST] The number of math problems that Marvin practiced today is three times as many as the number of problems he solved yesterday. His friend, Arvin, has practiced twice as many math problems on each day. How many math problems have they practiced altogether if Marvin solved 40 math problems yesterday?[/INST] If Marvin solved 40 math problems yesterday, he has solved 40*3 = <<40*3=120>>120 math problems today.
The total number of math problems that Marvin has practiced is 120+40 = <<120+40=...


# Accuracy

In [7]:
def extract_final_number(text):
    if text is None: return None
    if "####" in text:
        segment = text.split("####")[-1]
    else:
        segment = text

    nums = re.findall(r"-?\d+\.?\d*", segment)
    if nums: return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def gsm8k_accuracy(preds, golds):
    correct = 0
    rows = []
    for p, g in zip(preds, golds):
        pn = extract_final_number(p)
        ok = pn == str(g)
        correct += ok
        rows.append({"prediction": p, "pred_num": pn, "gold": g, "correct": ok})
    acc = correct / len(preds) if preds else 0
    return acc, pd.DataFrame(rows)

In [8]:
def generate_batch(model, tokenizer, questions, max_new_tokens=512):
    """Generate predictions for a batch of questions"""
    preds = []
    for q in tqdm(questions, desc="Generating"):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": q},
        ]
        prompt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        inp = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=max_new_tokens,
                temperature=0.2,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        txt = tokenizer.decode(
            out[0][inp["input_ids"].shape[-1]:],
            skip_special_tokens=True,
        ).strip()
        preds.append(txt)
    return preds

eval_questions = eval_df["question"].tolist()
eval_golds     = eval_df["gold_answer"].tolist()

# ZS Eval

In [9]:
print("Running ZERO-SHOT...")
FastLanguageModel.for_inference(model)
zs_preds = generate_batch(model, tokenizer, eval_questions)
zs_acc, zs_detail = gsm8k_accuracy(zs_preds, eval_golds)

print(f"\nZero-shot accuracy (n=500): {zs_acc:.4f}")
zs_detail.to_csv("eval_zero_shot_500.csv", index=False)

Running ZERO-SHOT...


Generating:   0%|          | 0/500 [00:00<?, ?it/s]


Zero-shot accuracy (n=500): 0.4120


# QLoRA SFT

In [10]:
model.train()
print("Applying LoRA adapters...")
model_sft = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

Applying LoRA adapters...


Unsloth 2025.12.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [11]:
MAX_STEPS=1200

train_args = SFTConfig(
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    
    # Batch size settings
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    
    # Learning rate
    learning_rate=2e-4,
    warmup_steps=10,
    
    # Training length
    max_steps=MAX_STEPS,
    num_train_epochs=1,
    
    # Optimization
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    
    # Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Logging and saving
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    
    # Output
    output_dir="./mistral_sft_lora",
    report_to="none",
    
    # Data processing
    dataset_num_proc=2,
    packing=False,
    
    seed=SEED,
)

trainer = SFTTrainer(
    model=model_sft,
    tokenizer=tokenizer,
    train_dataset=sft_text,
    args=train_args,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
print("Starting SFT training...")

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.\n")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print("Training Complete!")
print(f"{trainer_stats.metrics['train_runtime']:.2f} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

trainer.model.save_pretrained("mistral_sft_final")
tokenizer.save_pretrained("mistral_sft_final")
print("\nModel saved to mistral_sft_final/")

Starting SFT training...
GPU = Tesla T4. Max memory = 14.741 GB.
13.773 GB of memory reserved.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 3 | Total steps = 1,200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.408300
50,0.899400
75,0.894200
100,0.896100
125,0.871100
150,0.857700
175,0.841800
200,0.874300
225,0.881500
250,0.868500


Training Complete!
6416.52 seconds used for training.
106.94 minutes used for training.
Peak reserved memory = 14.184 GB.
Peak reserved memory for training = 0.411 GB.
Peak reserved memory % of max memory = 96.221 %.
Peak reserved memory for training % of max memory = 2.788 %.

Model saved to mistral_sft_final/


# Post SFT Eval

In [13]:
print("Evaluating SFT model...")
FastLanguageModel.for_inference(trainer.model)

ft_preds = generate_batch(trainer.model, tokenizer, eval_questions)
ft_acc, ft_detail = gsm8k_accuracy(ft_preds, eval_golds)

print(f"\nSFT accuracy (n=500): {ft_acc:.4f}")
print(f"Δ vs Zero-shot: {ft_acc - zs_acc:+.4f}")

ft_detail.to_csv("eval_sft_500.csv", index=False)

Evaluating SFT model...


Generating:   0%|          | 0/500 [00:00<?, ?it/s]


SFT accuracy (n=500): 0.5800
Δ vs Zero-shot: +0.1680


In [14]:
import shutil
shutil.make_archive("mistral_sft_lora",  "zip", "./mistral_sft_lora")
shutil.make_archive("mistral_sft_final", "zip", "./mistral_sft_final")
print("Archives created!")

Archives created!
